In [2]:
import glob, os, warnings
warnings.filterwarnings("ignore")
import multiprocessing as mp
import numpy as np
import pandas as pd

In [3]:
def merge_counts(cells, name, time, base, out_prefix):
    path1 = out_prefix + ".%s_based.filelist.txt" % base
    path2 = out_prefix + ".%s_based.csv" % base
    path3 = out_prefix + ".%s_based.annotated.csv" % base

    with open(path1, "w+") as fw:
        for cell in cells:
            if base == "gene":
                path = "../../1_NanoNASCseq/results/5_expression/4_quant_genes/min_read_2_min_tc_2/%s/%s.tsv" % (cell.split(".")[0], cell)
            else:
                if "MouseBlastocyst" in out_prefix:
                    path = "../../1_NanoNASCseq/results/8_expression_novel/2_quant_isoforms/min_read_2_min_tc_2/%s/%s.tsv" % (cell.split(".")[0], cell)
                else:
                    path = "../../1_NanoNASCseq/results/5_expression/6_quant_isoforms/min_read_2_min_tc_2/%s/%s.tsv" % (cell.split(".")[0], cell)
            fw.write(path + "\n")
            
    ! ../../1_NanoNASCseq/scripts/expression/merge_counts.py {path1} {path2}

    d = pd.read_csv(path2, index_col=0)
    d["TPM"] = d["Total"] * 1e6 / d["Total"].sum()
    d["TP10K"] = d["Total"] * 1e4 / d["Total"].sum()
    d["NTR"] = d["New"] / d["Total"]
    d["Halflife"] = np.nan if time == 0 else -time / np.log2(1 - d["NTR"])
    d["DecayRate"] = np.log(2) / d["Halflife"]
    d["SynthesisRate"] = d["DecayRate"] * d["TP10K"]
    
    if name.startswith("K562"):
        annfile = "/home/chenzonggui/species/homo_sapiens/GRCh38.p13/gencode.v39.%s_info.csv" % base
    elif name.startswith("mESC"):
        annfile = "/home/chenzonggui/species/mus_musculus/GRCm38.p6/gencode.vM25.%s_info.csv" % base
    elif name.startswith("MouseBlastocyst"):
        annfile = "../../1_NanoNASCseq/results/7_assembly_custom/5_gtf_full/MouseBlastocyst.gtf.%s_info.csv" % base
    else:
        assert False        
    d = d.merge(pd.read_csv(annfile, index_col=0), left_index=True, right_index=True)
    d.to_csv(path3)

    ! rm {path2}

In [4]:
def get_treatment_time(name):
    if "s4U_0uM" in name:
        return 0
    elif name.endswith("_3h") or name.endswith("_3h.highTC"):
        return 3
    elif name.endswith("_2h"):
        return 2
    elif name.endswith("_1h"):
        return 1
    elif name.endswith("_30min"):
        return 0.5
    elif name.endswith("_15min"):
        return 0.25
    else:
        print(name)
        return 0
        
pool = mp.Pool(24)
for path in sorted(glob.glob("results/tables/*.csv")):
    name = path.split("/")[-1][:-4]
    time = get_treatment_time(name)
    d = pd.read_csv(path, index_col=0)
    pool.apply_async(merge_counts, (d.index, name, time, "gene", "results/pseudobulk/%s" % name))
    pool.apply_async(merge_counts, (d.index, name, time, "transcript", "results/pseudobulk/%s" % name))
pool.close()
pool.join()

K562
mESC
